In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


In [2]:

# ------------------------
# Vision Transformer for MNIST
# ------------------------
class ViT_MNIST(nn.Module):
    def __init__(self, img_size=28, patch_size=7, d_model=64, nhead=4, num_layers=4, num_classes=10):
        super().__init__()
        assert img_size % patch_size == 0
        self.patch_size = patch_size
        num_patches = (img_size // patch_size) ** 2
        patch_dim = patch_size * patch_size

        # Patch embedding
        self.patch_embed = nn.Linear(patch_dim, d_model)

        # Class token & position embedding
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches + 1, d_model))

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # Classifier
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        B, C, H, W = x.shape
        # Split image into patches
        x = x.unfold(2, self.patch_size, self.patch_size).unfold(3, self.patch_size, self.patch_size)
        x = x.contiguous().view(B, C, -1, self.patch_size*self.patch_size)  # (B, C, num_patches, patch_dim)
        x = x.squeeze(1)  # MNIST: C=1 -> (B, num_patches, patch_dim)

        # Patch embedding
        x = self.patch_embed(x)

        # Add cls token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)

        # Add positional encoding
        x = x + self.pos_embed

        # Transformer encoding
        x = self.transformer(x)

        # Use CLS token for classification
        cls_out = x[:, 0]

        return self.fc(cls_out)

# ------------------------
# Training script
# ------------------------
def train_model(device="cpu", epochs=5, batch_size=64, lr=1e-3):
    # Data
    transform = transforms.Compose([
        transforms.ToTensor()
    ])
    train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
    test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size)

    # Model
    model = ViT_MNIST().to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Training loop
    for epoch in range(epochs):
        model.train()
        total_loss, correct = 0, 0
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()

        acc = correct / len(train_loader.dataset)
        print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader.dataset):.4f}, Acc: {acc:.4f}")

        # Evaluate
        model.eval()
        correct = 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs = model(imgs)
                correct += (outputs.argmax(1) == labels).sum().item()
        test_acc = correct / len(test_loader.dataset)
        print(f"  >> Test Acc: {test_acc:.4f}")




In [3]:
if __name__ == "__main__":
    device = "cuda" if torch.cuda.is_available() else "cpu"
    train_model(device=device, epochs=5)

Epoch 1/5, Loss: 2.3092, Acc: 0.1085
  >> Test Acc: 0.1623
Epoch 2/5, Loss: 2.2313, Acc: 0.1490
  >> Test Acc: 0.1135
Epoch 3/5, Loss: 2.3040, Acc: 0.1068
  >> Test Acc: 0.1135
Epoch 4/5, Loss: 2.3032, Acc: 0.1068
  >> Test Acc: 0.1135
Epoch 5/5, Loss: 2.3028, Acc: 0.1097
  >> Test Acc: 0.1135
